# SD Trainer Colab
Colab NoteBook Created by [licyk](https://github.com/licyk)

Jupyter Notebook 仓库：[licyk/sd-webui-all-in-one](https://github.com/licyk/sd-webui-all-in-one)

这是适用于 [Colab](https://colab.research.google.com) 部署 [SD-Trainer](https://github.com/Akegarasu/lora-scripts) / [Kohya GUI](https://github.com/bmaltais/kohya_ss) 的 Jupyter Notebook，使用时请按顺序执行 Jupyter Notebook 单元。

Colab 链接：<a href="https://colab.research.google.com/github/licyk/sd-webui-all-in-one/blob/main/sd_trainer_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 功能
1. 环境配置：配置安装的 PyTorch 版本、内网穿透的方式，并安装 SD Trainer。默认设置下已启用`配置环境完成后立即启动 SD Trainer`选项，则安装完成后将直接启动 SD Trainer，并显示访问地址。
2. 下载模型：下载可选列表中的模型（可选）。
3. 自定义模型下载：使用链接下载模型（可选）。
4. 扩展下载：使用链接下载扩展（可选）。
5. 启动 SD Trainer：启动 SD Trainer，并显示访问地址。


## 使用
1. 在 Colab -> 代码执行程序 > 更改运行时类型 -> 硬件加速器 选择`GPU T4`或者其他 GPU。
2. `环境配置`单元中的选项通常不需要修改，保持默认即可。如需切换其他 SD Trainer 分支，可修改`环境配置`单元中的`SD Trainer 分支预设`选项。
3. 运行`环境配置`单元，默认设置下已启用`配置环境完成后立即启动 SD Trainer`选项，则环境配置完成后立即启动 SD Trainer，此时将弹出 Google Drive 授权页面，根据提示进行操作。配置完成后将启动 SD Trainer，SD Trainer 的访问地址可在日志中查看。
4. 如果未启用`配置环境完成后立即启动 SD Trainer`选项，需要运行`启动 SD Trainer`单元，此时将弹出 Google Drive 授权页面，根据提示进行操作。配置完成后将启动 SD Trainer，SD Trainer 的访问地址可在日志中查看。
5. 提示词查询工具：[SD 绘画提示词查询](https://licyk.github.io/t/tag)


## 提示
1. Colab 在关机后将会清除所有数据，所以每次重新启动时必须运行`环境配置`单元才能运行`启动 SD Trainer`单元。
2. [Ngrok](https://ngrok.com) 内网穿透在使用前需要填写 Ngrok Token，可在 [Ngrok](https://ngrok.com) 官网获取。
3. [Zrok](https://zrok.io) 内网穿透在使用前需要填写 Zrok Token, 可在 [Zrok](https://docs.zrok.io/docs/getting-started) 官网获取。
4. 每次启动 Colab 后必须运行`环境配置`单元，才能运行`启动 SD Trainer`单元启动 SD Trainer。
5. 其他功能有自定义模型下载等功能，根据自己的需求进行使用。
6. 运行`启动 SD Trainer`时将弹出 Google Drive 访问授权提示，根据提示进行授权。授权后，使用 SD Trainer 生成的图片将保存在 Google Drive 的`sd_trainer_output`文件夹中。
7. 在`额外挂载目录设置`中可以挂载一些额外目录，如 LoRA 模型目录，挂载后的目录可在 Google Drive 的`sd_trainer_output`文件夹中查看和修改。可通过该功能在 Google Drive 上传模型并在 SD Trainer 中使用。
8. 默认挂载以下目录到 Google Drive，可在 Google Drive 的`sd_trainer_output`文件夹中上传和修改文件：`outputs`, `output`, `config`, `train`, `logs`

In [ ]:
# @title 👇 环境配置
# SD WebUI All In One 功能初始化部分, 通常不需要修改
# 如果需要查看完整代码实现, 可阅读: https://github.com/licyk/sd-webui-all-in-one/blob/main/sd_webui_all_in_one
#################################################################################################################
import os
import sys
import shutil

try:
    _ = JUPYTER_ROOT_PATH  # type: ignore # noqa: F821
except Exception:
    JUPYTER_ROOT_PATH = os.getcwd()
try:
    _ = INIT_SD_WEBUI_ALL_IN_ONE_CORE  # type: ignore # noqa: F821
    print("[Core] SD WebUI All In One 核心模块已初始化")
except Exception:
    print("[Core] 初始化 SD WebUI All In One 核心模块中")
    !python -m pip install sd-webui-all-in-one --upgrade
    from sd_webui_all_in_one import logger, VERSION, SDTrainerManager

    INIT_SD_WEBUI_ALL_IN_ONE_CORE = True
    logger.info("SD WebUI All In One 核心模块初始化完成, 版本: %s", VERSION)

#######################################################

# SD Trainer 分支预设
SD_TRAINER_BRANCH_PRESET_DICT = {
    "Akegarasu/SD-Trainer 分支": {
        "launch_args": "--skip-prepare-onnxruntime",
        "webui_port": 28000,
        "branch": "sd_trainer_main",
    },
    "bmaltais/Kohya GUI 分支": {
        "launch_args": "--inbrowser --language zh-CN --noverify",
        "webui_port": 7860,
        "branch": "kohya_gui_main",
    },
}

# @markdown ## SD Trainer 核心配置选项
# @markdown - SD Trainer 安装路径：
SD_TRAINER_PATH = "/content/sd-trainer"  # @param {type:"string"}
# @markdown - SD Trainer 分支预设：
SD_TRAINER_BRANCH_PRESET = "Akegarasu/SD-Trainer 分支"  # @param ["Akegarasu/SD-Trainer 分支", "bmaltais/Kohya GUI 分支"]
# @markdown - SD Trainer 启动参数（可选，不填则使用分支预设值）：
SD_TRAINER_LAUNCH_ARGS_OPT = ""  # @param {type:"string"}
# @markdown - SD Trainer 访问端口（可选，不填则使用分支预设值）：
SD_TRAINER_WEBUI_PORT_OPT = ""  # @param {type:"string"}
# @markdown ---
# @markdown ## PyTorch 组件版本选项：
# @markdown - 安装 PyTorch 的类型：
PYTORCH_DEVICE_TYPE = "auto"  # @param ["auto", "cu113", "cu117", "cu118", "cu121", "cu124", "cu126", "cu128", "cu129", "cu130", "rocm5.4.2", "rocm5.6", "rocm5.7", "rocm6.0", "rocm6.1", "rocm6.2", "rocm6.2.4", "rocm6.3", "rocm6.4", "rocm7.1", "xpu", "ipex_legacy_arc", "cpu", "directml", "all"]
# @markdown - PyTorch：
PYTORCH_VER = ""  # @param {type:"string"}
# @markdown - xFormers：
XFORMERS_VER = ""  # @param {type:"string"}
# @markdown ---
# @markdown ## 包管理器选项：
# @markdown - 使用 uv 作为 Python 包管理器
USE_UV = True  # @param {type:"boolean"}
# @markdown ---
# @markdown ## 镜像源选项：
# @markdown - 是否使用 PyPI 镜像
USE_PYPI_MIRROR = False  # @param {type:"boolean"}
# @markdown - PyPI 主镜像源
PIP_INDEX_MIRROR = "https://pypi.python.org/simple"  # @param {type:"string"}
# @markdown - PyPI 扩展镜像源
PIP_EXTRA_INDEX_MIRROR = "https://download.pytorch.org/whl/cu126"  # @param {type:"string"}
# @markdown - PyPI 额外镜像源
PIP_FIND_LINKS_MIRROR = ""  # @param {type:"string"}
# @markdown - 是否使用 Github 镜像
USE_GITHUB_MIRROR = False  # @param {type:"boolean"}
# @markdown - Github 镜像地址
CUSTOM_GITHUB_MIRROR = ""  # @param {type:"string"}
# @markdown - 是否使用 HuggingFace 镜像
USE_HF_MIRROR = False  # @param {type:"boolean"}
# @markdown - HuggingFace 镜像源地址
HUGGINGFACE_MIRROR = ""  # @param {type:"string"}
# @markdown - 是否使用国内模型镜像
USE_CN_MODEL_MIRROR = False  # @param {type:"boolean"}
# @markdown ---
# @markdown ## 内网穿透选项：
# @markdown - 使用 remote.moe 内网穿透
USE_REMOTE_MOE = True  # @param {type:"boolean"}
# @markdown - 使用 localhost.run 内网穿透
USE_LOCALHOST_RUN = True  # @param {type:"boolean"}
# @markdown - 使用 pinggy.io 内网穿透
USE_PINGGY_IO = True  # @param {type:"boolean"}
# @markdown - 使用 CloudFlare 内网穿透
USE_CLOUDFLARE = True  # @param {type:"boolean"}
# @markdown - 使用 Gradio 内网穿透
USE_GRADIO = True  # @param {type:"boolean"}
# @markdown - 使用 Ngrok 内网穿透（需填写 Ngrok Token，可在 [Ngrok](https://ngrok.com) 官网获取）
USE_NGROK = True  # @param {type:"boolean"}
# @markdown - Ngrok Token
NGROK_TOKEN = ""  # @param {type:"string"}
# @markdown - 使用 Zrok 内网穿透（需填写 Zrok Token，可在 [Zrok](https://docs.zrok.io/docs/getting-started) 官网获取）
USE_ZROK = True  # @param {type:"boolean"}
# @markdown - Zrok Token
ZROK_TOKEN = ""  # @param {type:"string"}
# @markdown ---
# @markdown ## 快速启动选项：
# @markdown - 配置环境完成后立即启动 ComfyUI（并挂载 Google Drive）
QUICK_LAUNCH = True  # @param {type:"boolean"}
# @markdown - 不重复配置环境（当重复运行环境配置时将不会再进行安装）
NO_REPEAT_CONFIGURE_ENV = True  # @param {type:"boolean"}
# @markdown ---
# @markdown ## 其他选项：
# @markdown - 清理无用日志
CLEAR_LOG = True  # @param {type:"boolean"}
# @markdown - 检查可用 GPU
CHECK_AVALIABLE_GPU = True  # @param {type:"boolean"}
# @markdown - 启用 TCMalloc 内存优化
ENABLE_TCMALLOC = True  # @param {type:"boolean"}
# @markdown - 启用 CUDA Malloc 显存优化
ENABLE_CUDA_MALLOC = True  # @param {type:"boolean"}
# @markdown - 出现冲突组件时按顺序安装组件依赖
INSTALL_CONFLICT_COMPONENT = True  # @param {type:"boolean"}
# @markdown - 更新内核和扩展
UPDATE_CORE = True  # @param {type:"boolean"}
# @markdown -  HuggingFace Token
HUGGINGFACE_TOKEN = ""  # @param {type:"string"}
# @markdown -  ModelScope Token
MODELSCOPE_TOKEN = ""  # @param {type:"string"}

# 预设参数处理
SD_TRAINER_LAUNCH_ARGS = SD_TRAINER_LAUNCH_ARGS_OPT or SD_TRAINER_BRANCH_PRESET_DICT.get(SD_TRAINER_BRANCH_PRESET).get("launch_args")
SD_TRAINER_WEBUI_PORT = SD_TRAINER_WEBUI_PORT_OPT or SD_TRAINER_BRANCH_PRESET_DICT.get(SD_TRAINER_BRANCH_PRESET).get("webui_port")
INSTALL_BRANCH = SD_TRAINER_BRANCH_PRESET_DICT.get("branch")

# @markdown ---
##############################################################################
# @markdown ## 模型设置, 在安装时将会下载选择的模型：
SD_MODEL = []

# @markdown - Stable Diffusion 模型
v1_5_pruned_emaonly = False  # @param  {type:"boolean"}
animefull_final_pruned = False  # @param  {type:"boolean"}
sd_xl_base_1_0_0_9vae = False  # @param  {type:"boolean"}
sd_xl_refiner_1_0_0_9vae = False  # @param  {type:"boolean"}
sd_xl_turbo_1_0_fp16 = False  # @param  {type:"boolean"}
animagine_xl_3_0_base = False  # @param  {type:"boolean"}
animagine_xl_3_0 = False  # @param  {type:"boolean"}
animagine_xl_3_1 = False  # @param  {type:"boolean"}
animagine_xl_4_0 = False  # @param  {type:"boolean"}
animagine_xl_4_0_opt = False  # @param  {type:"boolean"}
holodayo_xl_2_1 = False  # @param  {type:"boolean"}
kivotos_xl_2_0 = False  # @param  {type:"boolean"}
clandestine_xl_1_0 = False  # @param  {type:"boolean"}
UrangDiffusion_1_1 = False  # @param  {type:"boolean"}
RaeDiffusion_XL_v2 = False  # @param  {type:"boolean"}
kohaku_xl_delta_rev1 = False  # @param  {type:"boolean"}
kohakuXLEpsilon_rev1 = False  # @param  {type:"boolean"}
kohaku_xl_epsilon_rev2 = False  # @param  {type:"boolean"}
kohaku_xl_epsilon_rev3 = False  # @param  {type:"boolean"}
kohaku_xl_zeta = False  # @param  {type:"boolean"}
starryXLV52_v52 = False  # @param  {type:"boolean"}
heartOfAppleXL_v20 = False  # @param  {type:"boolean"}
heartOfAppleXL_v30 = False  # @param  {type:"boolean"}
sanaexlAnimeV10_v10 = False  # @param  {type:"boolean"}
sanaexlAnimeV10_v11 = False  # @param  {type:"boolean"}
SanaeXL_Anime_v1_2_aesthetic = False  # @param  {type:"boolean"}
SanaeXL_Anime_v1_3_aesthetic = False  # @param  {type:"boolean"}
Illustrious_XL_v0_1 = False  # @param  {type:"boolean"}
Illustrious_XL_v0_1_GUIDED = False  # @param  {type:"boolean"}
Illustrious_XL_v1_0 = False  # @param  {type:"boolean"}
Illustrious_XL_v1_1 = False  # @param  {type:"boolean"}
Illustrious_XL_v2_0_stable = False  # @param  {type:"boolean"}
jruTheJourneyRemains_v25XL = False  # @param  {type:"boolean"}
noobaiXLNAIXL_earlyAccessVersion = False  # @param  {type:"boolean"}
noobaiXLNAIXL_epsilonPred05Version = False  # @param  {type:"boolean"}
noobaiXLNAIXL_epsilonPred075 = False  # @param  {type:"boolean"}
noobaiXLNAIXL_epsilonPred077 = False  # @param  {type:"boolean"}
noobaiXLNAIXL_epsilonPred10Version = False  # @param  {type:"boolean"}
noobaiXLNAIXL_epsilonPred11Version = False  # @param  {type:"boolean"}
noobaiXLNAIXL_vPredTestVersion = False  # @param  {type:"boolean"}
noobaiXLNAIXL_vPred05Version = False  # @param  {type:"boolean"}
noobaiXLNAIXL_vPred06Version = False  # @param  {type:"boolean"}
noobaiXLNAIXL_vPred065SVersion = False  # @param  {type:"boolean"}
noobaiXLNAIXL_vPred075SVersion = False  # @param  {type:"boolean"}
noobaiXLNAIXL_vPred09RVersion = False  # @param  {type:"boolean"}
noobaiXLNAIXL_vPred10Version = True  # @param  {type:"boolean"}
ChenkinNoob_XL_V01 = False  # @param  {type:"boolean"}
ponyDiffusionV6XL_v6StartWithThisOne = False  # @param  {type:"boolean"}
pdForAnime_v20 = False  # @param  {type:"boolean"}
omegaPonyXLAnime_v20 = False  # @param  {type:"boolean"}
# @markdown - VAE 模型
vae_ft_ema_560000_ema_pruned = False  # @param  {type:"boolean"}
vae_ft_mse_840000_ema_pruned = False  # @param  {type:"boolean"}
sdxl_fp16_fix_vae = True  # @param  {type:"boolean"}


# Stable Diffusion 模型
v1_5_pruned_emaonly and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sd_1.5/v1-5-pruned-emaonly.safetensors"})
animefull_final_pruned and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sd_1.5/animefull-final-pruned.safetensors"})
sd_xl_base_1_0_0_9vae and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/sd_xl_base_1.0_0.9vae.safetensors"})
sd_xl_refiner_1_0_0_9vae and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/sd_xl_refiner_1.0_0.9vae.safetensors"})
sd_xl_turbo_1_0_fp16 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/sd_xl_turbo_1.0_fp16.safetensors"})
animagine_xl_3_0_base and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/animagine-xl-3.0-base.safetensors"})
animagine_xl_3_0 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/animagine-xl-3.0.safetensors"})
animagine_xl_3_1 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/animagine-xl-3.1.safetensors"})
animagine_xl_4_0 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/animagine-xl-4.0.safetensors"})
animagine_xl_4_0_opt and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/animagine-xl-4.0-opt.safetensors"})
holodayo_xl_2_1 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/holodayo-xl-2.1.safetensors"})
kivotos_xl_2_0 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kivotos-xl-2.0.safetensors"})
clandestine_xl_1_0 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/clandestine-xl-1.0.safetensors"})
UrangDiffusion_1_1 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/UrangDiffusion-1.1.safetensors"})
RaeDiffusion_XL_v2 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/RaeDiffusion-XL-v2.safetensors"})
kohaku_xl_delta_rev1 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kohaku-xl-delta-rev1.safetensors"})
kohakuXLEpsilon_rev1 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kohakuXLEpsilon_rev1.safetensors"})
kohaku_xl_epsilon_rev2 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kohaku-xl-epsilon-rev2.safetensors"})
kohaku_xl_epsilon_rev3 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kohaku-xl-epsilon-rev3.safetensors"})
kohaku_xl_zeta and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kohaku-xl-zeta.safetensors"})
starryXLV52_v52 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/starryXLV52_v52.safetensors"})
heartOfAppleXL_v20 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/heartOfAppleXL_v20.safetensors"})
heartOfAppleXL_v30 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/heartOfAppleXL_v30.safetensors"})
sanaexlAnimeV10_v10 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/sanaexlAnimeV10_v10.safetensors"})
sanaexlAnimeV10_v11 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/sanaexlAnimeV10_v11.safetensors"})
SanaeXL_Anime_v1_2_aesthetic and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/SanaeXL-Anime-v1.2-aesthetic.safetensors"})
SanaeXL_Anime_v1_3_aesthetic and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/SanaeXL-Anime-v1.3-aesthetic.safetensors"})
Illustrious_XL_v0_1 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/Illustrious-XL-v0.1.safetensors"})
Illustrious_XL_v0_1_GUIDED and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/Illustrious-XL-v0.1-GUIDED.safetensors"})
Illustrious_XL_v1_0 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/Illustrious-XL-v1.0.safetensors"})
Illustrious_XL_v1_1 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/Illustrious-XL-v1.1.safetensors"})
Illustrious_XL_v2_0_stable and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/Illustrious-XL-v2.0-stable.safetensors"})
jruTheJourneyRemains_v25XL and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/jruTheJourneyRemains_v25XL.safetensors"})
noobaiXLNAIXL_earlyAccessVersion and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_earlyAccessVersion.safetensors"})
noobaiXLNAIXL_epsilonPred05Version and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_epsilonPred05Version.safetensors"})
noobaiXLNAIXL_epsilonPred075 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_epsilonPred075.safetensors"})
noobaiXLNAIXL_epsilonPred077 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_epsilonPred077.safetensors"})
noobaiXLNAIXL_epsilonPred10Version and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_epsilonPred10Version.safetensors"})
noobaiXLNAIXL_epsilonPred11Version and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_epsilonPred11Version.safetensors"})
noobaiXLNAIXL_vPredTestVersion and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPredTestVersion.safetensors"})
noobaiXLNAIXL_vPred05Version and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred05Version.safetensors"})
noobaiXLNAIXL_vPred06Version and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred06Version.safetensors"})
noobaiXLNAIXL_vPred065SVersion and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred065SVersion.safetensors"})
noobaiXLNAIXL_vPred075SVersion and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred075SVersion.safetensors"})
noobaiXLNAIXL_vPred09RVersion and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred09RVersion.safetensors"})
noobaiXLNAIXL_vPred10Version and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred10Version.safetensors"})
ChenkinNoob_XL_V01 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/ChenkinNoob-XL-V0.1.safetensors"})
ponyDiffusionV6XL_v6StartWithThisOne and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/ponyDiffusionV6XL_v6StartWithThisOne.safetensors"})
pdForAnime_v20 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/pdForAnime_v20.safetensors"})
omegaPonyXLAnime_v20 and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/omegaPonyXLAnime_v20.safetensors"})
# VAE 模型
vae_ft_ema_560000_ema_pruned and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-vae/resolve/main/sd_1.5/vae-ft-ema-560000-ema-pruned.safetensors"})
vae_ft_mse_840000_ema_pruned and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-vae/resolve/main/sd_1.5/vae-ft-mse-840000-ema-pruned.safetensors"})
sdxl_fp16_fix_vae and SD_MODEL.append({"url": "https://huggingface.co/licyk/sd-vae/resolve/main/sdxl_1.0/sdxl_fp16_fix_vae.safetensors"})

# @markdown ---
##############################################################################
# @markdown ## 额外挂载目录设置, 在启动时挂载目录到 Google Drive：
EXTRA_LINK_DIR = []

# @markdown - config_files 目录
link_config_files_dir = False  # @param {type:"boolean"}
# @markdown - dataset 目录
link_dataset_dir = False  # @param {type:"boolean"}
# @markdown - localizations 目录
link_localizations_dir = False  # @param {type:"boolean"}
# @markdown - presets 目录
link_presets_dir = False  # @param {type:"boolean"}


link_config_files_dir and EXTRA_LINK_DIR.append({"link_dir": "config_files"})
link_dataset_dir and EXTRA_LINK_DIR.append({"link_dir": "dataset"})
link_localizations_dir and EXTRA_LINK_DIR.append({"link_dir": "localizations"})
link_presets_dir and EXTRA_LINK_DIR.append({"link_dir": "presets"})

##############################################################################

if PYTORCH_DEVICE_TYPE == "auto":
    PYTORCH_DEVICE_TYPE = None

INSTALL_PARAMS = {
    "pytorch_mirror_type": PYTORCH_DEVICE_TYPE,
    "custom_pytorch_package": PYTORCH_VER or None,
    "custom_xformers_package": XFORMERS_VER or None,
    "use_pypi_mirror": USE_PYPI_MIRROR,
    "use_uv": USE_UV,
    "use_github_mirror": USE_GITHUB_MIRROR,
    "custom_github_mirror": CUSTOM_GITHUB_MIRROR or None,
    "install_branch": INSTALL_BRANCH,
    "no_pre_download_model": True,  # 使用 model_list 参数替换
    "use_cn_model_mirror": USE_CN_MODEL_MIRROR,
    "use_hf_mirror": USE_HF_MIRROR,
    "pypi_index_mirror": PIP_INDEX_MIRROR or None,
    "pypi_extra_index_mirror": PIP_EXTRA_INDEX_MIRROR or None,
    "pypi_find_links_mirror": PIP_FIND_LINKS_MIRROR or None,
    "github_mirror": CUSTOM_GITHUB_MIRROR or None,
    "huggingface_mirror": HUGGINGFACE_MIRROR or None,
    "model_list": SD_MODEL,
    "check_avaliable_gpu": CHECK_AVALIABLE_GPU,
    "enable_tcmalloc": ENABLE_TCMALLOC,
    "enable_cuda_malloc": ENABLE_CUDA_MALLOC,
    "custom_sys_pkg_cmd": None if sys.platform == "linux" and shutil.which("apt") else [],
    "huggingface_token": HUGGINGFACE_TOKEN or None,
    "modelscope_token": MODELSCOPE_TOKEN or None,
    "update_core": UPDATE_CORE,
}

CHECK_PARAMS = {
    "use_uv": USE_UV,
    "use_github_mirror": USE_GITHUB_MIRROR,
    "custom_github_mirror": CUSTOM_GITHUB_MIRROR or None,
    "use_pypi_mirror": USE_PYPI_MIRROR,
}

TUNNEL_PARAMS = {
    "use_ngrok": USE_NGROK,
    "ngrok_token": NGROK_TOKEN or None,
    "use_cloudflare": USE_CLOUDFLARE,
    "use_remote_moe": USE_REMOTE_MOE,
    "use_localhost_run": USE_LOCALHOST_RUN,
    "use_gradio": USE_GRADIO,
    "use_pinggy_io": USE_PINGGY_IO,
    "use_zrok": USE_ZROK,
    "zrok_token": ZROK_TOKEN or None,
    "webui_name": "SD Trainer",
}

try:
    _ = sd_trainer_manager_has_init  # type: ignore  # noqa: F821
except Exception:
    sd_trainer = SDTrainerManager(os.path.dirname(SD_TRAINER_PATH), os.path.basename(SD_TRAINER_PATH), port=int(SD_TRAINER_WEBUI_PORT))
    sd_trainer_manager_has_init = True

try:
    _ = sd_trainer_has_init  # type: ignore  # noqa: F821
except Exception:
    sd_trainer_has_init = False

if NO_REPEAT_CONFIGURE_ENV:
    if not sd_trainer_has_init:
        sd_trainer.install(**INSTALL_PARAMS)
        INIT_CONFIG = 1
        sd_trainer_has_init = True
        CLEAR_LOG and sd_trainer.clear_output()
        logger.info("SD Trainer 运行环境配置完成")
    else:
        logger.info("检测到不重复配置环境已启用, 并且在当前 Colab 实例中已配置 SD Trainer 运行环境, 不再重复配置 SD Trainer 运行环境")
        logger.info("如需在当前 Colab 实例中重新配置 SD Trainer 运行环境, 请在快速启动选项中取消不重复配置环境功能")
else:
    sd_trainer.install(**INSTALL_PARAMS)
    INIT_CONFIG = 1
    CLEAR_LOG and sd_trainer.clear_output()
    logger.info("SD Trainer 运行环境配置完成")

LAUNCH_CMD = sd_trainer.get_launch_command(SD_TRAINER_LAUNCH_ARGS)

if QUICK_LAUNCH:
    logger.info("启动 SD Trainer 中")
    os.chdir(SD_TRAINER_PATH)
    sd_trainer.check_env(**CHECK_PARAMS)
    sd_trainer.mount_drive(EXTRA_LINK_DIR)
    sd_trainer.get_tunnel_url(**TUNNEL_PARAMS)
    logger.info("SD Trainer 启动参数: %s", SD_TRAINER_LAUNCH_ARGS)
    !{LAUNCH_CMD}
    os.chdir(JUPYTER_ROOT_PATH)
    CLEAR_LOG and sd_trainer.clear_output()
    logger.info("已关闭 SD Trainer")

In [ ]:
#@title 👇 自定义模型下载
try:
    _ = INIT_CONFIG
except Exception:
    raise Exception("未安装 SD Trainer")

#@markdown ### 填写模型的下载链接：
model_url = "https://huggingface.co/licyk/sd-lora/resolve/main/sdxl/style/CoolFlatColor.safetensors"  #@param {type:"string"}
#@markdown ### 填写模型的名称（包括后缀名）：
model_name = "CoolFlatColor.safetensors"  #@param {type:"string"}

sd_trainer.get_sd_model(
    url=model_url,
    filename=model_name or None,
)

In [ ]:
#@title 👇 启动 SD Trainer
try:
    _ = INIT_CONFIG
except Exception:
    raise Exception("未安装 SD Trainer")

logger.info("启动 SD Trainer")
os.chdir(SD_TRAINER_PATH)
sd_trainer.check_env(**CHECK_PARAMS)
sd_trainer.mount_drive(EXTRA_LINK_DIR)
sd_trainer.get_tunnel_url(**TUNNEL_PARAMS)
logger.info("SD Trainer 启动参数: %s", SD_TRAINER_LAUNCH_ARGS)
!{LAUNCH_CMD}
os.chdir(JUPYTER_ROOT_PATH)
CLEAR_LOG and sd_trainer.clear_output()
logger.info("已关闭 SD Trainer")

In [ ]:
#@title 👇 文件下载工具

#@markdown ### 填写文件的下载链接：
url = ""  #@param {type:"string"}
#@markdown ### 填写文件的保存路径：
file_path = "/content"  #@param {type:"string"}
#@markdown ### 填写文件的保存名称 (可选)：
filename = ""  #@param {type:"string"}

sd_trainer.download_file_from_url(
    url=url,
    path=file_path or None,
    save_name=filename or None,
)
